# Fraud Risk Summary

Reads `payments.card_transactions` (see
`data-generators/card_transactions_ingest.ipynb`) and summarizes fraud
risk by transaction amount range and hour of day — the kind of
risk-scoring rollup a payments/banking fraud team monitors to spot where
risk concentrates.

`Time` in the source is seconds elapsed since the first transaction, not
a wall-clock timestamp — it's converted to an hour-of-day bucket by
taking it modulo one day.

Re-runnable: `mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "payments"
SOURCE_TABLE = "card_transactions"
TARGET_TABLE = "fraud_risk_summary_v1"

## Load bronze table

In [ ]:
df = spark.table(f"{DATABASE_NAME}.{SOURCE_TABLE}")

## Transform: bucket by amount range and hour of day

In [ ]:
bucketed_df = (
    df
    .withColumn("hour_of_day", (F.col("Time").cast("long") % 86400 / 3600).cast("int"))
    .withColumn(
        "amount_bucket",
        F.when(F.col("Amount") < 10, "0-10")
        .when(F.col("Amount") < 50, "10-50")
        .when(F.col("Amount") < 100, "50-100")
        .when(F.col("Amount") < 500, "100-500")
        .otherwise("500+"),
    )
    .withColumn("is_fraud", F.when(F.col("Class") == "1", 1).otherwise(0))
)

## Aggregate by amount bucket and hour

In [ ]:
summary_df = (
    bucketed_df
    .groupBy("amount_bucket", "hour_of_day")
    .agg(
        F.count("*").alias("transaction_count"),
        F.sum("is_fraud").alias("fraud_count"),
        F.round(F.sum(F.when(F.col("is_fraud") == 1, F.col("Amount")).otherwise(0.0)), 2).alias("total_flagged_amount"),
    )
    .withColumn("fraud_rate_pct", F.round(F.col("fraud_count") * 100.0 / F.col("transaction_count"), 3))
    .orderBy(F.desc("fraud_count"))
)

## Persist as a managed Spark table

In [ ]:
summary_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TARGET_TABLE}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_buckets FROM {DATABASE_NAME}.{TARGET_TABLE}").show()
spark.sql(f"SELECT * FROM {DATABASE_NAME}.{TARGET_TABLE} ORDER BY fraud_count DESC LIMIT 10").show()